# Fusion — `baselines.ipynb` fixed

Safe replacement for the original baseline notebook.

Fixes:
- Stream B/C are loaded before Stream A label repair.
- `verite_123` and `mmfake_123` are never parsed as binary labels.
- Feature concatenation requires key alignment or exact label-matching row alignment.
- Output CSV/figure/cache filenames are unchanged.

## 1. Imports & paths

In [ ]:
import os, sys, json, pickle, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

try:
    ROOT = Path(__file__).resolve().parents[2]
except NameError:
    ROOT = Path(os.getcwd()).resolve()
    while not (ROOT / "src" / "utils" / "paths.py").exists():
        if ROOT == ROOT.parent:
            ROOT = Path(os.getcwd()).resolve()
            break
        ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))
print(f"Project root: {ROOT}")

def load_pkl(path):
    with open(path, "rb") as f:
        return pickle.load(f)

def save_pkl(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(obj, f, protocol=4)
    print(f"Saved {path}")

def as_int(y):
    return np.asarray(y).reshape(-1).astype(int)

def binary(y):
    return set(np.unique(as_int(y)).tolist()).issubset({0, 1})

def key_str(k):
    return k.decode("utf-8", errors="replace") if isinstance(k, bytes) else str(k)

def safe_auc(y, p):
    from sklearn.metrics import roc_auc_score
    y = as_int(y); p = np.asarray(p)
    return np.nan if len(np.unique(y)) < 2 else roc_auc_score(y, p)

from sklearn.metrics import roc_auc_score, accuracy_score, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

ENT_DIR    = ROOT / "cache" / "entity_scorer"
SB_DIR     = ROOT / "cache" / "stream_b_final"
SC_DIR     = ROOT / "cache" / "stance_scores"
FUSION_DIR = ROOT / "cache" / "fusion"
FIG_DIR    = ROOT / "paper" / "figures"
TAB_DIR    = ROOT / "paper" / "tables"
for d in (FUSION_DIR, FIG_DIR, TAB_DIR):
    d.mkdir(parents=True, exist_ok=True)

## 2. Load Stream B and Stream C first

In [ ]:
SB_FILES = {
    "train":  SB_DIR / "stream_b_final_nc_train.pkl",
    "val":    SB_DIR / "stream_b_final_nc_val.pkl",
    "test":   SB_DIR / "stream_b_final_nc_test.pkl",
    "verite": SB_DIR / "stream_b_final_verite.pkl",
    "mmfake": SB_DIR / "stream_b_final_mmfake.pkl",
}
SC_FILES = {
    "train":  SC_DIR / "stream_c_train.pkl",
    "val":    SC_DIR / "stream_c_val.pkl",
    "test":   SC_DIR / "stream_c_test.pkl",
    "verite": SC_DIR / "stream_c_verite.pkl",
    "mmfake": SC_DIR / "stream_c_mmfake.pkl",
}

def normalise_bundle(d, split, stream):
    X = np.asarray(d["features"], dtype=np.float32)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    y = as_int(d["labels"])
    if len(X) != len(y):
        raise ValueError(f"{stream} {split}: X/y length mismatch: {len(X)} vs {len(y)}")
    keys = d.get("pair_keys", [f"{split}_row_{i}" for i in range(len(y))])
    keys = [key_str(k) for k in keys]
    feat_names = d.get("feat_names", [f"{stream.lower().replace(' ', '_')}_{i}" for i in range(X.shape[1])])
    return {
        "features": np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0),
        "labels": y,
        "pair_keys": keys,
        "feat_names": list(feat_names),
        "source": stream,
    }

sb, sc_data = {}, {}
for label, mapping, target in [("Stream B", SB_FILES, sb), ("Stream C", SC_FILES, sc_data)]:
    print(label)
    for split, path in mapping.items():
        if not path.exists():
            print(f"  MISSING {split}: {path}")
            continue
        target[split] = normalise_bundle(load_pkl(path), split, label)
        y = target[split]["labels"]
        print(f"  {split:8s}: X={target[split]['features'].shape}, pos={y.sum()}/{len(y)}")

## 3. Load Stream A with safe labels

In [ ]:
SA_FILES = {
    "train":  ENT_DIR / "features_newsclippings_train.pkl",
    "val":    ENT_DIR / "features_newsclippings_val.pkl",
    "test":   ENT_DIR / "features_newsclippings_test.pkl",
    "verite": ENT_DIR / "features_verite.pkl",
    "mmfake": ENT_DIR / "features_mmfake.pkl",
}

def parse_label_from_key(k):
    s = str(k)
    if "_" not in s:
        return None
    prefix, suffix = s.rsplit("_", 1)
    if suffix not in {"0", "1"}:
        return None
    if prefix.lower().startswith(("verite", "mmfake")):
        return None
    return int(suffix)

def stream_a_to_bundle(data, split):
    if isinstance(data, pd.DataFrame):
        label_col = next((c for c in ("label", "falsified", "y", "target") if c in data.columns), None)
        blocked = {"label", "falsified", "y", "target", "pair_key", "pair_id", "image_id", "split"}
        feat_cols = [c for c in data.columns if c not in blocked and pd.api.types.is_numeric_dtype(data[c])]
        X = data[feat_cols].to_numpy(dtype=np.float32)
        y = as_int(data[label_col].to_numpy()) if label_col else np.full(len(X), -1, dtype=int)
        keys = data["pair_key"].tolist() if "pair_key" in data.columns else [f"{split}_row_{i}" for i in range(len(X))]
    elif isinstance(data, dict) and "features" in data:
        X = np.asarray(data["features"], dtype=np.float32)
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        y = as_int(data["labels"]) if "labels" in data else np.full(len(X), -1, dtype=int)
        keys = data.get("pair_keys", [f"{split}_row_{i}" for i in range(len(X))])
        feat_cols = data.get("feat_names", [f"sa_{i}" for i in range(X.shape[1])])
    elif isinstance(data, dict):
        keys = list(data.keys())
        sample = next(iter(data.values()))
        blocked = {"label", "falsified", "y", "target", "pair_key", "pair_id", "image_id", "split"}
        feat_cols = [k for k, v in sample.items() if k not in blocked and isinstance(v, (int, float, np.integer, np.floating, bool))]
        X = np.array([[data[k].get(f, 0.0) for f in feat_cols] for k in keys], dtype=np.float32)
        label_key = next((c for c in ("label", "falsified", "y", "target") if c in sample), None)
        if label_key:
            y = np.array([data[k].get(label_key, -1) for k in keys], dtype=int)
        else:
            parsed = [parse_label_from_key(k) for k in keys]
            y = np.array(parsed, dtype=int) if all(v is not None for v in parsed) else np.full(len(keys), -1, dtype=int)
    else:
        raise TypeError(f"Unsupported Stream A format: {type(data)}")

    keys = [key_str(k) for k in keys]
    if len(X) != len(keys):
        raise ValueError(f"Stream A {split}: X/key mismatch")
    return {"features": np.nan_to_num(X), "labels": y, "pair_keys": keys, "feat_names": list(feat_cols), "source": "Stream A"}

sa = {}
for split, path in SA_FILES.items():
    if not path.exists():
        print(f"MISSING {split}: {path}")
        continue
    b = stream_a_to_bundle(load_pkl(path), split)
    # Repair OOD labels only from same-length trusted streams.
    if not binary(b["labels"]):
        for ref_name, ref in [("Stream B", sb), ("Stream C", sc_data)]:
            if split in ref and len(ref[split]["labels"]) == len(b["labels"]):
                b["labels"] = ref[split]["labels"].copy()
                print(f"  {split}: repaired Stream A labels from {ref_name}")
                break
    if not binary(b["labels"]):
        raise ValueError(f"Stream A {split}: labels are still not binary: {np.unique(b['labels'])[:10]}")
    sa[split] = b
    y = b["labels"]
    print(f"  {split:8s}: X={b['features'].shape}, pos={y.sum()}/{len(y)}")

## 4. Alignment helpers

In [ ]:
def try_key_align(split, bundles):
    maps = {}
    common = None
    for name, b in bundles.items():
        keys = b["pair_keys"]
        if len(keys) != len(set(keys)):
            raise ValueError(f"{split}: duplicate keys in {name}")
        m = {k: i for i, k in enumerate(keys)}
        maps[name] = m
        common = set(m) if common is None else common & set(m)
    if not common:
        raise ValueError(f"{split}: no common keys")

    ref = bundles["Stream A"]["pair_keys"] if "Stream A" in bundles else next(iter(bundles.values()))["pair_keys"]
    ordered = [k for k in ref if k in common]
    out = {}
    for name, b in bundles.items():
        idx = np.array([maps[name][k] for k in ordered])
        out[name] = {**b, "features": b["features"][idx], "labels": b["labels"][idx], "pair_keys": ordered}
    ys = [b["labels"] for b in out.values()]
    if not all(np.array_equal(ys[0], y) for y in ys[1:]):
        raise ValueError(f"{split}: labels mismatch after key alignment")
    return out, f"key alignment n={len(ordered)}"

def try_row_align(split, bundles):
    lengths = {n: len(b["labels"]) for n, b in bundles.items()}
    if len(set(lengths.values())) != 1:
        raise ValueError(f"{split}: lengths differ {lengths}")
    ys = [b["labels"] for b in bundles.values()]
    if not all(np.array_equal(ys[0], y) for y in ys[1:]):
        raise ValueError(f"{split}: labels differ and no key alignment was possible")
    n = len(ys[0])
    keys = [f"{split}_row_{i}" for i in range(n)]
    return {name: {**b, "pair_keys": keys} for name, b in bundles.items()}, f"row alignment n={n}"

def align_split(split, require=("Stream A", "Stream B", "Stream C")):
    src = {}
    if "Stream A" in require: src["Stream A"] = sa[split]
    if "Stream B" in require: src["Stream B"] = sb[split]
    if "Stream C" in require: src["Stream C"] = sc_data[split]
    try:
        return try_key_align(split, src)
    except Exception as e1:
        try:
            out, msg = try_row_align(split, src)
            return out, msg + f" (key align failed: {e1})"
        except Exception as e2:
            raise ValueError(f"{split}: unsafe to align. key error={e1}; row error={e2}")

print("Alignment report")
alignment_report = {}
for split in ("train", "val", "test", "verite", "mmfake"):
    try:
        aligned, method = align_split(split)
        alignment_report[split] = {"ok": True, "method": method, "n": len(aligned["Stream A"]["labels"])}
        print(f"  {split:8s}: OK {method}")
    except Exception as e:
        alignment_report[split] = {"ok": False, "error": str(e)}
        print(f"  {split:8s}: NOT SAFE - {e}")

## 5. Individual LR probes

In [ ]:
def fit_lr(train_bundle):
    scaler = StandardScaler()
    X = scaler.fit_transform(train_bundle["features"])
    y = train_bundle["labels"]
    lr = LogisticRegression(max_iter=2000, C=1.0, random_state=42)
    lr.fit(X, y)
    return lr, scaler

def predict_lr(model, scaler, bundle):
    p = model.predict_proba(scaler.transform(bundle["features"]))[:, 1]
    return {"auroc": safe_auc(bundle["labels"], p), "acc": accuracy_score(bundle["labels"], (p >= 0.5).astype(int)),
            "probs": p, "labels": bundle["labels"], "pair_keys": bundle["pair_keys"]}

individual_results = {}
stream_probs = {"Stream A": {}, "Stream B": {}, "Stream C": {}}
models = {}
for name, data in [("Stream A", sa), ("Stream B", sb), ("Stream C", sc_data)]:
    if "train" not in data:
        continue
    lr, scaler = fit_lr(data["train"])
    models[name] = (lr, scaler)
    for split in ("train", "val", "test", "verite", "mmfake"):
        if split not in data:
            continue
        res = predict_lr(lr, scaler, data[split])
        individual_results[(name, split)] = res
        stream_probs[name][split] = res["probs"]
        print(f"{name:8s} {split:8s}: AUROC={res['auroc']:.4f} Acc={res['acc']:.4f} N={len(res['labels'])}")

## 6. Safe concatenated feature fusion

In [ ]:
def concat_bundle(split):
    aligned, method = align_split(split)
    X = np.hstack([aligned["Stream A"]["features"], aligned["Stream B"]["features"], aligned["Stream C"]["features"]]).astype(np.float32)
    y = aligned["Stream A"]["labels"]
    names = ([f"A::{n}" for n in aligned["Stream A"]["feat_names"]] +
             [f"B::{n}" for n in aligned["Stream B"]["feat_names"]] +
             [f"C::{n}" for n in aligned["Stream C"]["feat_names"]])
    return {"features": X, "labels": y, "pair_keys": aligned["Stream A"]["pair_keys"], "feat_names": names,
            "aligned": aligned, "alignment_method": method,
            "dims": {"stream_a": aligned["Stream A"]["features"].shape[1],
                     "stream_b": aligned["Stream B"]["features"].shape[1],
                     "stream_c": aligned["Stream C"]["features"].shape[1]}}

concat_results, concat_bundles = {}, {}
try:
    train_concat = concat_bundle("train")
    concat_bundles["train"] = train_concat
    concat_lr, concat_scaler = fit_lr(train_concat)
    for split in ("val", "test", "verite", "mmfake"):
        try:
            b = concat_bundle(split)
            concat_bundles[split] = b
            res = predict_lr(concat_lr, concat_scaler, b)
            res["alignment_method"] = b["alignment_method"]
            concat_results[split] = res
            print(f"Concat {split:8s}: AUROC={res['auroc']:.4f} via {b['alignment_method']}")
        except Exception as e:
            print(f"Concat {split:8s}: SKIPPED - {e}")
except Exception as e:
    concat_lr = None
    print(f"Concat fusion skipped: {e}")

## 7. Probability fusion and stacking

In [ ]:
def same_labels(*items):
    if any(x is None for x in items): return False
    if len(set(len(x["labels"]) for x in items)) != 1: return False
    return all(np.array_equal(items[0]["labels"], x["labels"]) for x in items[1:])

prob_fusion_results = {}
for split in ("test", "verite", "mmfake"):
    items = [individual_results.get((s, split)) for s in ("Stream A", "Stream B", "Stream C")]
    if not same_labels(*items):
        print(f"Probability fusion {split}: skipped (stream labels/lengths differ)")
        continue
    y = items[0]["labels"]
    pa, pb, pc = [x["probs"] for x in items]
    prob_fusion_results[split] = {
        "mean": safe_auc(y, (pa + pb + pc) / 3),
        "max": safe_auc(y, np.maximum(np.maximum(pa, pb), pc)),
        "weighted": safe_auc(y, 0.2 * pa + 0.6 * pb + 0.2 * pc),
    }
    print(f"Prob fusion {split}: {prob_fusion_results[split]}")

stacking_results = {}
train_items = [individual_results.get((s, "train")) for s in ("Stream A", "Stream B", "Stream C")]
if same_labels(*train_items):
    Xs = np.column_stack([x["probs"] for x in train_items])
    ys = train_items[0]["labels"]
    stack_lr = LogisticRegression(max_iter=2000, C=1.0, random_state=42).fit(Xs, ys)
    for split in ("val", "test", "verite", "mmfake"):
        items = [individual_results.get((s, split)) for s in ("Stream A", "Stream B", "Stream C")]
        if not same_labels(*items):
            print(f"Stacking {split}: skipped")
            continue
        X = np.column_stack([x["probs"] for x in items])
        p = stack_lr.predict_proba(X)[:, 1]
        stacking_results[split] = {"auroc": safe_auc(items[0]["labels"], p),
                                   "acc": accuracy_score(items[0]["labels"], (p >= 0.5).astype(int)),
                                   "probs": p, "labels": items[0]["labels"], "pair_keys": items[0]["pair_keys"]}
        print(f"Stacking {split:8s}: AUROC={stacking_results[split]['auroc']:.4f}")
else:
    stack_lr = None
    print("Stacking skipped: train stream probabilities are not safely aligned.")

## 8. Save same CSV/fig/cache filenames

In [ ]:
ALL_RESULTS = {
    "Majority":    {"NC test": 0.500, "VERITE": 0.500, "MMFake": 0.500},
    "Stream A":    {},
    "Stream B":    {},
    "Stream C":    {},
    "Mean fusion": {},
    "Weighted":    {},
    "Stacking":    {},
    "Concat LR":   {},
}
SPLIT_MAP = {"test": "NC test", "verite": "VERITE", "mmfake": "MMFake"}

for split, col in SPLIT_MAP.items():
    for s in ("Stream A", "Stream B", "Stream C"):
        if (s, split) in individual_results:
            ALL_RESULTS[s][col] = individual_results[(s, split)]["auroc"]
    if split in prob_fusion_results:
        ALL_RESULTS["Mean fusion"][col] = prob_fusion_results[split]["mean"]
        ALL_RESULTS["Weighted"][col] = prob_fusion_results[split]["weighted"]
    if split in stacking_results:
        ALL_RESULTS["Stacking"][col] = stacking_results[split]["auroc"]
    if split in concat_results:
        ALL_RESULTS["Concat LR"][col] = concat_results[split]["auroc"]

df_all = pd.DataFrame(ALL_RESULTS).T.round(4)
df_all.index.name = "Method"
display(df_all)
df_all.to_csv(TAB_DIR / "fusion_baselines.csv")
print("Saved paper/tables/fusion_baselines.csv")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (split, title) in zip(axes, [("test", "NewsCLIPpings test"), ("verite", "VERITE"), ("mmfake", "MMFakeBench")]):
    for s in ("Stream A", "Stream B", "Stream C"):
        r = individual_results.get((s, split))
        if r is None or len(np.unique(r["labels"])) < 2:
            continue
        fpr, tpr, _ = roc_curve(r["labels"], r["probs"])
        ax.plot(fpr, tpr, label=f"{s} ({r['auroc']:.3f})")
    for label, resdict in [("Concat LR", concat_results), ("Stacking", stacking_results)]:
        r = resdict.get(split)
        if r is not None and len(np.unique(r["labels"])) >= 2:
            fpr, tpr, _ = roc_curve(r["labels"], r["probs"])
            ax.plot(fpr, tpr, linestyle="--", label=f"{label} ({r['auroc']:.3f})")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
    ax.set_title(title); ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "fusion_baselines_roc.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved paper/figures/fusion_baselines_roc.png")

for split, b in concat_bundles.items():
    a = b["aligned"]
    bundle = {
        "stream_a": a["Stream A"]["features"], "stream_b": a["Stream B"]["features"], "stream_c": a["Stream C"]["features"],
        "labels": b["labels"], "pair_keys": b["pair_keys"],
        "sa_feat_names": a["Stream A"]["feat_names"], "sb_dim": a["Stream B"]["features"].shape[1], "sc_dim": a["Stream C"]["features"].shape[1],
        "alignment_method": b["alignment_method"], "dims": b["dims"],
        "prob_a": stream_probs["Stream A"].get(split), "prob_b": stream_probs["Stream B"].get(split), "prob_c": stream_probs["Stream C"].get(split),
    }
    save_pkl(bundle, FUSION_DIR / f"fusion_input_{split}.pkl")

save_pkl({
    "individual": {str(k): float(v["auroc"]) for k, v in individual_results.items()},
    "concat": {k: float(v["auroc"]) for k, v in concat_results.items()},
    "stacking": {k: float(v["auroc"]) for k, v in stacking_results.items()},
    "prob_fusion": prob_fusion_results,
    "stack_coef": None if stack_lr is None else stack_lr.coef_[0].tolist(),
    "alignment_report": alignment_report,
}, FUSION_DIR / "baseline_results.pkl")